# Steam Game Reviews
## Notebook 05 — Retrieval-Augmented Generation

Owner: Mitsos · local (Apple Silicon)

Task 2: a game recommendation assistant that answers free-text questions grounded
in real player reviews.

Pipeline: question → Sentence-BERT query embedding → FAISS top-20 → cross-encoder
rerank to top-5 → prompt with numbered evidence → Groq-hosted LLM → answer.

Steps:
1. Setup and artifacts
2. Build the retrieval corpus
3. FAISS index
4. Retriever and cross-encoder reranking
5. Prompt engineering
6. Generation
7. Evaluation: retrieval hit-rate, reranking ablation, groundedness
8. Generator comparison
9. Export for the Streamlit app

Design decisions. We do not chunk: classic RAG splits long documents into passages,
but reviews are already passage-sized, so each filtered review is one retrieval
unit. Retrieval is two-stage because the bi-encoder is cheap but coarse and the
cross-encoder is precise but slow, so we retrieve wide and rerank narrow.


## 1. Setup and artifacts

The Groq key is read from a `.env` file in the project root — never hardcoded, and
`.env` is in `.gitignore`. Create one containing `GROQ_API_KEY=gsk_...` from a free
key at console.groq.com.

In [1]:
from pathlib import Path
import json
import os
import time
import textwrap
import warnings

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import CrossEncoder, SentenceTransformer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"

SEED = 42
# Retrieval-corpus settings
MIN_WORDS = 50                    # drop reviews shorter than this from the retrieval corpus
NEAR_DUPLICATE_THRESHOLD = 0.95   # cosine similarity above which two reviews count as duplicates
RETRIEVE_K = 20                   # candidates fetched by the fast bi-encoder stage
FINAL_K = 5                       # evidence reviews kept after cross-encoder reranking

PRIMARY_MODEL = "llama-3.3-70b-versatile"   # deployed default: fast
ALTERNATE_MODEL = "openai/gpt-oss-120b"     # comparison: slower, more detailed
# Groq rotates its lineup — verify both at console.groq.com/docs/models before a demo.

def find_project_dir(markers=("Data", "data", "outputs"), max_up=4):
    """Locate the project root by walking up from the working directory.

    Portable across macOS, Windows and Linux: no hard-coded absolute paths, and it
    works whether the notebook sits in the project root or in a subfolder.
    """
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents][: max_up + 1]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError(
        f"Could not locate the project root starting from {start}.\n"
        f"Expected a folder containing one of {markers}.\n"
        "Open the notebook from inside the project folder (the one holding Data/)."
    )

PROJECT_DIR = find_project_dir()

OUTPUT_DIR = PROJECT_DIR / "outputs"
EMBEDDING_DIR = OUTPUT_DIR / "embeddings"
RAG_DIR = OUTPUT_DIR / "rag"
TABLE_DIR = OUTPUT_DIR / "tables"
RAG_DIR.mkdir(parents=True, exist_ok=True)

# CUDA (Windows/Linux + NVIDIA) -> MPS (Apple Silicon) -> CPU.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Load the embedding matrix and row-aligned metadata written by notebook 03
embeddings = np.load(EMBEDDING_DIR / "sbert_minilm_embeddings.npy")
metadata = pd.read_csv(EMBEDDING_DIR / "sbert_row_metadata.csv", low_memory=False)
assert len(embeddings) == len(metadata), "Embeddings and metadata are out of sync"

# Read the Groq key from .env (VS Code kernels do not inherit the shell environment)
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_DIR / ".env")
except ImportError:
    print("python-dotenv not installed; relying on the shell environment.")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "").strip()

print("Device:", DEVICE)
print("Embedding matrix:", embeddings.shape)
print("Groq key available:", bool(GROQ_API_KEY))

/opt/homebrew/Caskroom/miniforge/base/envs/nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Embedding matrix: (120000, 384)
Groq key available: True


## 2. Retrieval corpus

Two quality filters, applied to the retrieval corpus only — the classification
experiments are untouched.

A length floor of 50 words: shorter reviews ("10/10", "good game") carry no usable
evidence and would dilute every result set.

Near-duplicate removal: Steam has copy-pasted meme reviews, and without this the
top-5 can be the same text five times. Detection runs iteratively — removing one
member of a duplicate pair can expose new pairs — until no pair above the
similarity threshold remains.


In [2]:
# Keep only reviews long enough to serve as evidence
long_mask = metadata["word_count"].ge(MIN_WORDS).to_numpy()
# Display/prompt text uses model_text (leading-year artifact removed in notebook 00),
# renamed to `review` so everything downstream is unchanged. The embeddings were
# already built from model_text in notebook 03, so this only affects what is shown.
corpus = metadata.loc[long_mask, [
    "raw_row_id", "game_name", "recommendation", "model_text", "word_count", "primary_genre"
]].rename(columns={"model_text": "review"}).reset_index(drop=True)
# Slice the matching embedding rows and L2-normalise so inner product = cosine
corpus_embeddings = embeddings[long_mask].astype("float32", copy=True)
faiss.normalize_L2(corpus_embeddings)

# Remove exact duplicates: identical text posted under the same game
before_exact = len(corpus)
exact_keep = ~corpus.duplicated(["game_name", "review"], keep="first")
corpus = corpus.loc[exact_keep].reset_index(drop=True)
corpus_embeddings = corpus_embeddings[exact_keep.to_numpy()]

print("Reviews with at least", MIN_WORDS, "words:", f"{before_exact:,}")
print("Exact same-game duplicates removed:", f"{before_exact - len(corpus):,}")

Reviews with at least 50 words: 41,720
Exact same-game duplicates removed: 26


In [3]:
# Iterative near-duplicate removal: each pass finds every review's nearest neighbour
# and drops one member of any pair above the threshold; repeat until no pair remains
near_duplicate_removed = 0
for iteration in range(1, 21):
    temporary_index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
    temporary_index.add(corpus_embeddings)
    # k=2 because each vector's nearest neighbour is itself (column 0); column 1 is the real one
    scores, neighbours = temporary_index.search(corpus_embeddings, 2)
    # Of each duplicate pair, keep the lower row id and remove the higher
    remove_ids = {
        max(row_id, int(neighbour_id))
        for row_id, (similarity, neighbour_id) in enumerate(zip(scores[:, 1], neighbours[:, 1]))
        if similarity > NEAR_DUPLICATE_THRESHOLD and neighbour_id >= 0 and row_id != neighbour_id
    }
    if not remove_ids:
        break
    keep_mask = np.ones(len(corpus), dtype=bool)
    keep_mask[list(remove_ids)] = False
    corpus = corpus.loc[keep_mask].reset_index(drop=True)
    corpus_embeddings = corpus_embeddings[keep_mask]
    near_duplicate_removed += len(remove_ids)
    print(f"  pass {iteration}: removed {len(remove_ids):,}")

print("Near-duplicates removed:", f"{near_duplicate_removed:,}")
print("Final RAG corpus:", f"{len(corpus):,}", "reviews from", corpus["game_name"].nunique(), "games")

  pass 1: removed 465
  pass 2: removed 45
  pass 3: removed 12
  pass 4: removed 2
Near-duplicates removed: 524
Final RAG corpus: 41,170 reviews from 241 games


## 3. FAISS index

Vectors are L2-normalised, so inner product is cosine similarity and
`IndexFlatIP` gives exact search. At this corpus size an exact index is instant;
approximate indexes (IVF, HNSW) only start paying off in the millions.

In [4]:
# Exact inner-product index; on normalised vectors this is exact cosine search
index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)
print("Index:", type(index).__name__, "|", index.ntotal, "vectors x", index.d, "dims")
print("Index/corpus aligned:", index.ntotal == len(corpus))

Index: IndexFlatIP | 41170 vectors x 384 dims
Index/corpus aligned: True


## 4. Retriever and reranking

The bi-encoder compresses query and review into vectors independently — fast, but
lossy. A cross-encoder reads the pair together and scores actual relevance; too
slow for the whole corpus, ideal for reordering 20 candidates. Section 7 measures
what this second stage is worth.

In [5]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
reranker_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Bi-encoder for the wide first stage, cross-encoder for the precise second stage
embedding_model = SentenceTransformer(embedding_model_name, device=DEVICE)
reranker = CrossEncoder(reranker_model_name, device=DEVICE)

# Stage 1: embed the question and fetch the k nearest reviews from the index
def retrieve(query, k=RETRIEVE_K):
    query_vector = embedding_model.encode([query], normalize_embeddings=True,
                                          convert_to_numpy=True).astype("float32")
    scores, row_ids = index.search(query_vector, min(k, len(corpus)))
    result = corpus.iloc[row_ids[0]].copy().reset_index(drop=True)
    result.insert(0, "bi_score", scores[0].round(3))
    return result

# Stage 2: score each (question, review) pair together and keep the best final_k
def retrieve_and_rerank(query, retrieve_k=RETRIEVE_K, final_k=FINAL_K):
    candidates = retrieve(query, k=retrieve_k)
    pairs = list(zip([query] * len(candidates), candidates["review"].tolist()))
    candidates["rerank_score"] = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
    return candidates.sort_values("rerank_score", ascending=False).head(final_k).reset_index(drop=True)

# Try a few example queries
for query in ["a relaxing game to play with friends",
              "an emotional story that made players cry",
              "games with poor PC performance or crashes"]:
    print("\nQUERY:", query)
    display(retrieve_and_rerank(query)[["game_name", "recommendation", "bi_score", "rerank_score"]])

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7630.76it/s]



QUERY: a relaxing game to play with friends


,game_name,recommendation,bi_score,rerank_score
0,Human Fall Flat,Recommended,0.660,7.767699
1,Stardew Valley,Recommended,0.659,7.095517
2,Tabletop Simulator,Recommended,0.656,6.630713
3,Crime Scene Cleaner,Recommended,0.649,6.405164
4,Party Animals,Recommended,0.676,6.289912



QUERY: an emotional story that made players cry


,game_name,recommendation,bi_score,rerank_score
0,It Takes Two Friend's Pass,Recommended,0.483,3.009483
1,OMORI,Recommended,0.608,1.392588
2,Total War: WARHAMMER,Recommended,0.465,0.556693
3,It Takes Two Friend's Pass,Recommended,0.516,0.008855
4,OMORI,Recommended,0.558,-0.486629



QUERY: games with poor PC performance or crashes


,game_name,recommendation,bi_score,rerank_score
0,Dorfromantik,Not Recommended,0.707,5.491202
1,STAR WARS Jedi: Survivor™,Recommended,0.663,5.005161
2,For The King,Not Recommended,0.664,4.448004
3,STAR WARS Jedi: Survivor™,Not Recommended,0.701,4.354959
4,F1® Manager 2024,Not Recommended,0.739,4.334244


## 5. Prompt engineering

The system prompt sets the role, restricts the model to the supplied evidence,
requires citing the bracketed evidence number, forbids presenting a negative
review as a recommendation, and instructs the model to admit when the evidence
does not support an answer. The user prompt packs the retrieved reviews as
numbered blocks with game name and verdict, so every claim is traceable.

In [6]:
SYSTEM_PROMPT = """You are a game recommendation assistant. Answer questions using ONLY the supplied player-review evidence.
- Base every game-specific claim on the evidence and cite its bracketed number.
- Never invent games or reviewer opinions.
- Do not present a negative review as a recommendation.
- If the evidence is insufficient, say so explicitly.
- Keep the answer concise: two to four short paragraphs."""

# Pack the evidence as numbered blocks with game name and verdict, then append the question
def build_prompt(query, evidence):
    blocks = []
    for position, row in evidence.reset_index(drop=True).iterrows():
        # Collapse whitespace and cap each review at 900 characters to keep the prompt bounded
        review_text = " ".join(str(row["review"]).split())[:900]
        blocks.append(f"[{position + 1}] Game: {row['game_name']}\n"
                      f"Player verdict: {row['recommendation']}\n"
                      f"Review evidence: {review_text}")
    context = "\n\n".join(blocks)
    return f"Player-review evidence:\n\n{context}\n\nQuestion: {query}\n\nAnswer only from the evidence above."

print(build_prompt("a relaxing game after work", retrieve_and_rerank("a relaxing game after work", final_k=3))[:1400])

Player-review evidence:

[1] Game: House Flipper
Player verdict: Recommended
Review evidence: Amazing and relaxing game. This is just what i needed when i feel stress after work or with family. If i want just to paint will paint, if i just want to buy furniture i will do that. Very satisfying game to play and to keep your anxiety and depression at check a tittle. Games like this and fishing and euro truck simulator are best for me and relaxing.

[2] Game: Dorfromantik
Player verdict: Recommended
Review evidence: I love how relaxing this game is! I wind up playing it a bunch after work to calm down. It’s not too demanding but you can make it as demanding as you like while playing. Tbf I often prefer to just try and make the prettiest looking town 😅 But even playing like this I have unlocked so many achievements!

[3] Game: Dorfromantik
Player verdict: Recommended
Review evidence: This game is great for the casual gamer that just wants to have a fun, stress-free evening after work. Serio

## 6. Generation

In [7]:
if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY missing. Add it to .env in the project root and restart the kernel.")

# Hosted generation client
from groq import Groq
client = Groq(api_key=GROQ_API_KEY)

# Send system prompt + packed evidence to the LLM and return its answer
def generate_answer(question, evidence, model_name=PRIMARY_MODEL,
                    temperature=0.2, max_tokens=500):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": build_prompt(question, evidence)}],
        temperature=temperature, max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Full pipeline for one question: retrieve, rerank, generate; optionally show the evidence
def rag_answer(question, model_name=PRIMARY_MODEL, final_k=FINAL_K, verbose=True):
    evidence = retrieve_and_rerank(question, final_k=final_k)
    answer = generate_answer(question, evidence, model_name=model_name)
    if verbose:
        print("QUESTION:", question)
        print("\nEVIDENCE:")
        for position, row in evidence.iterrows():
            print(f"  [{position + 1}] {row['game_name']} ({row['recommendation']}) "
                  f"rerank={row['rerank_score']:.2f}")
        print("\nANSWER:\n" + textwrap.fill(answer, width=100))
    return answer, evidence

for question in [
    "I want a relaxing game I can play in short sessions after work.",
    "Which games do players say have serious performance problems on PC?",
    "What do players say about monetization in free-to-play games?",
]:
    rag_answer(question)
    print("\n" + "=" * 100 + "\n")
    time.sleep(2)

QUESTION: I want a relaxing game I can play in short sessions after work.

EVIDENCE:
  [1] House Flipper (Recommended) rerank=4.86
  [2] Dorfromantik (Recommended) rerank=4.86
  [3] PowerWash Simulator (Recommended) rerank=4.06
  [4] PowerWash Simulator (Recommended) rerank=2.56
  [5] Stardew Valley (Recommended) rerank=1.86

ANSWER:
Based on the player-review evidence, several games are recommended for relaxation and can be played
in short sessions. For example, House Flipper [1] is described as "amazing and relaxing" and can be
played in a way that suits the player's mood, such as just painting or buying furniture.
Dorfromantik [2] is also recommended for its calm and enjoyable gameplay, with the added benefit of
being easy to play while doing something else, like watching a show or listening to music.   Stardew
Valley [5] is another option, described as "very calm and relaxing" and can be picked up and put
down whenever, making it suitable for short sessions.   PowerWash Simulator [

## 7. Evaluation

Three measurements on eight hand-written questions: retrieval hit-rate@5, the
reranking ablation, and a programmatic groundedness check that verifies whether
games named in an answer actually appear in the supplied evidence.

One design decision to note: the expected games for each question are derived from
the corpus itself — a game counts as expected if its reviews contain the question's
topic terms — rather than listed from memory, so the retriever is never penalised
for games that are not in the filtered corpus at all. Results and their
interpretation are discussed in the report.


In [8]:
# A game is "expected" for a question if its reviews contain the topic terms;
# take the n games with the most matching reviews
def games_for(pattern, n=8):
    matched = corpus["review"].str.contains(pattern, case=False, na=False, regex=True)
    return corpus.loc[matched, "game_name"].value_counts().head(n).index.tolist()

# Eight hand-written questions with corpus-derived expected games
evaluation_set = [
    {"question": "Which games do players say run badly or crash on PC?",
     "expected": games_for(r"crash|unoptimi|stutter|low fps|performance issue")},
    {"question": "What is a good cozy relaxing game?",
     "expected": games_for(r"relax|cozy|cosy|chill|calming|wholesome")},
    {"question": "Which games have an emotional story that made players cry?",
     "expected": games_for(r"\bcry\b|emotional|made me tear|heartbreak|made me feel")},
    {"question": "What do players think about grinding in online games?",
     "expected": games_for(r"grind|grinding")},
    {"question": "Which shooter games do players recommend for playing with friends?",
     "expected": games_for(r"shooter|gunplay|fps game")},
    {"question": "What do reviews say about kernel level anticheat?",
     "expected": games_for(r"anticheat|anti-cheat|kernel")},
    {"question": "Which strategy games are worth buying according to players?",
     "expected": games_for(r"strategy|tactical|\brts\b|turn-based|turn based")},
    {"question": "What sports or racing games do players enjoy most?",
     "expected": games_for(r"racing|\bsports\b|football|soccer|driving sim")},
]

print("Corpus-derived expected games:")
for item in evaluation_set:
    print(f"  {item['question'][:52]:54s} {item['expected'][:3]}")

# 1 if any expected game appears in the top five results, else 0
def hit_at_five(retrieved_games, expected_games):
    return int(bool(set(retrieved_games[:5]) & set(expected_games)))

# Score every question twice: bi-encoder alone, and with the reranker
rows = []
for item in evaluation_set:
    if not item["expected"]:
        continue
    plain = retrieve(item["question"], k=5)["game_name"].tolist()
    reranked = retrieve_and_rerank(item["question"], final_k=5)["game_name"].tolist()
    rows.append({"question": item["question"][:60],
                 "n_expected": len(item["expected"]),
                 "hit@5 (bi-encoder)": hit_at_five(plain, item["expected"]),
                 "hit@5 (+ reranker)": hit_at_five(reranked, item["expected"])})

retrieval_evaluation = pd.DataFrame(rows)
retrieval_evaluation.to_csv(TABLE_DIR / "rag_retrieval_eval.csv", index=False)
display(retrieval_evaluation)
print("Bi-encoder hit@5:", f"{retrieval_evaluation['hit@5 (bi-encoder)'].mean():.0%}")
print("With reranking:  ", f"{retrieval_evaluation['hit@5 (+ reranker)'].mean():.0%}")

Corpus-derived expected games:
  Which games do players say run badly or crash on PC?   ['STAR WARS Jedi: Survivor™', 'ARK: Survival Ascended', 'Cities: Skylines II']
  What is a good cozy relaxing game?                     ['Dorfromantik', 'Stardew Valley', 'Fields of Mistria']
  Which games have an emotional story that made player   ['Life is Strange - Episode 1', 'NieR:Automata™', 'Ori and the Will of the Wisps']
  What do players think about grinding in online games   ['Warframe', 'The First Descendant', 'War Thunder']
  Which shooter games do players recommend for playing   ['Titanfall® 2', 'TROUBLESHOOTER: Abandoned Children', 'Fallout 4']
  What do reviews say about kernel level anticheat?      ['EA SPORTS™ WRC', 'Counter-Strike 2', 'Grand Theft Auto V']
  Which strategy games are worth buying according to p   ['Northgard', 'Stellaris', 'Symphony of War: The Nephilim Saga']
  What sports or racing games do players enjoy most?     ['Assetto Corsa Competizione', 'Assetto Corsa', '

,question,n_expected,hit@5 (bi-encoder),hit@5 (+ reranker)
0,Which games do players say run badly or crash ...,8,1,1
1,What is a good cozy relaxing game?,8,1,1
2,Which games have an emotional story that made ...,8,1,1
3,What do players think about grinding in online...,8,1,1
4,Which shooter games do players recommend for p...,8,1,1
5,What do reviews say about kernel level anticheat?,8,1,1
6,Which strategy games are worth buying accordin...,8,0,0
7,What sports or racing games do players enjoy m...,8,1,1


Bi-encoder hit@5: 88%
With reranking:   88%


In [9]:
# Which games does the answer mention, and are they all present in the evidence?
# Matching is by name substring, so it is conservative on franchise/DLC naming
def grounding_check(answer, evidence):
    evidence_games = set(evidence["game_name"])
    all_games = set(corpus["game_name"].unique())
    mentioned = {game for game in all_games if game.lower() in answer.lower()}
    unsupported = mentioned - evidence_games
    return {"games_mentioned": len(mentioned),
            "grounded": len(mentioned & evidence_games),
            "hallucinated": len(unsupported),
            "hallucinated_names": sorted(unsupported)}

# Generate an answer for five questions and check each one
grounding_rows = []
for item in evaluation_set[:5]:
    answer, evidence = rag_answer(item["question"], verbose=False)
    row = grounding_check(answer, evidence)
    row["question"] = item["question"][:55]
    grounding_rows.append(row)
    time.sleep(2)

grounding = pd.DataFrame(grounding_rows)[
    ["question", "games_mentioned", "grounded", "hallucinated", "hallucinated_names"]]
grounding.to_csv(TABLE_DIR / "rag_groundedness.csv", index=False)
display(grounding)
print(f"Hallucination rate: {grounding['hallucinated'].sum()} / {grounding['games_mentioned'].sum()} games mentioned")

,question,games_mentioned,grounded,hallucinated,hallucinated_names
0,Which games do players say run badly or crash ...,2,2,0,[]
1,What is a good cozy relaxing game?,2,2,0,[]
2,Which games have an emotional story that made ...,5,4,1,[It Takes Two]
3,What do players think about grinding in online...,5,5,0,[]
4,Which shooter games do players recommend for p...,4,4,0,[]


Hallucination rate: 1 / 18 games mentioned


## 8. Generator comparison

Retrieval hit-rate is measured before generation, so it does not depend on the LLM.
To assess the generation stage on its own, the same questions run through two
Groq-hosted models on identical retrieved evidence — any difference is then
attributable to the generator alone. The comparison and the deployment choice are
discussed in the report.


In [10]:
comparison_questions = [item["question"] for item in evaluation_set[:5]]

# Run all comparison questions through one model and collect its grounding stats
def evaluate_generator(model_name):
    rows, answers = [], {}
    for question in comparison_questions:
        answer, evidence = rag_answer(question, model_name=model_name, verbose=False)
        row = grounding_check(answer, evidence)
        row["question"] = question[:55]
        rows.append(row)
        answers[question] = answer
        time.sleep(2)
    return pd.DataFrame(rows), answers

print("Running", PRIMARY_MODEL, "...")
primary_frame, primary_answers = evaluate_generator(PRIMARY_MODEL)
print("Running", ALTERNATE_MODEL, "...")
alternate_frame, alternate_answers = evaluate_generator(ALTERNATE_MODEL)

# Side-by-side totals for the two generators on identical evidence
generator_summary = pd.DataFrame([
    {"model": PRIMARY_MODEL,
     "games_mentioned": int(primary_frame["games_mentioned"].sum()),
     "grounded": int(primary_frame["grounded"].sum()),
     "hallucinated": int(primary_frame["hallucinated"].sum())},
    {"model": ALTERNATE_MODEL,
     "games_mentioned": int(alternate_frame["games_mentioned"].sum()),
     "grounded": int(alternate_frame["grounded"].sum()),
     "hallucinated": int(alternate_frame["hallucinated"].sum())},
])
generator_summary.to_csv(TABLE_DIR / "rag_model_comparison.csv", index=False)
display(generator_summary)

Running llama-3.3-70b-versatile ...
Running openai/gpt-oss-120b ...


,model,games_mentioned,grounded,hallucinated
0,llama-3.3-70b-versatile,17,16,1
1,openai/gpt-oss-120b,8,7,1


In [11]:
# Read all five side by side: one example is not enough to judge a stylistic difference.
for question in comparison_questions:
    print("QUESTION:", question)
    print("=" * 96)
    print(f"{PRIMARY_MODEL}:\n")
    print(textwrap.fill(primary_answers[question], width=96))
    print("\n" + "-" * 96)
    print(f"{ALTERNATE_MODEL}:\n")
    print(textwrap.fill(alternate_answers[question], width=96))
    print("=" * 96 + "\n")

QUESTION: Which games do players say run badly or crash on PC?
llama-3.3-70b-versatile:

Based on the player-review evidence, the following games are reported to run badly or crash on
PC:  Hogwarts Legacy [1, 5] and Dorfromantik [2].  Additionally, theHunter: Call of the Wild [4]
is also reported to crash frequently on PC.  It's worth noting that one reviewer [3] reports
that Hogwarts Legacy no longer crashes after a patch was released, but other reviewers [1, 5]
still experience crashes.

------------------------------------------------------------------------------------------------
openai/gpt-oss-120b:

Players report serious stability and performance problems on PC for several
titles. Hogwarts Legacy is described as “absolutely sucks” on PC with constant crashes and
terrible performance even on high‑end hardware [1][5]. Dorfromantik exhausts memory, crashes the
game and even triggers Windows blue‑screens, with “horrendous performance” on default
settings [2]. theHunter: Call of the

## 9. Export for the Streamlit app

`rag_config.json` lists both generators under `groq_models` plus a `default_model`,
so the app can offer a dropdown. Retrieval does not depend on the generator —
switching models changes only the final call, the evidence is identical — which is
why the switcher is a few lines in the UI rather than a second pipeline.

On Hugging Face Spaces the Groq key goes into the Space secrets, never the repo.

In [12]:
# Save the index and corpus for the Streamlit app
faiss.write_index(index, str(RAG_DIR / "faiss_index.bin"))
corpus.to_csv(RAG_DIR / "rag_corpus.csv", index=False)

# App configuration: both generators are listed so the UI can offer a model switcher
config = {
    "embedding_model": embedding_model_name,
    "reranker_model": reranker_model_name,
    "groq_models": {
        "Llama 3.3 70B (fast)": PRIMARY_MODEL,
        "GPT-OSS 120B (detailed, cited)": ALTERNATE_MODEL,
    },
    "default_model": PRIMARY_MODEL,
    "retrieve_k": RETRIEVE_K,
    "final_k": FINAL_K,
    "min_words": MIN_WORDS,
    "near_duplicate_threshold": NEAR_DUPLICATE_THRESHOLD,
    "source_modelling_sample_reviews": int(len(metadata)),
    "rag_corpus_reviews": int(len(corpus)),
    "system_prompt": SYSTEM_PROMPT,
}
with open(RAG_DIR / "rag_config.json", "w") as handle:
    json.dump(config, handle, indent=2)

print("Saved:", *[p.name for p in sorted(RAG_DIR.iterdir())])
print("Models offered:", list(config["groq_models"]))

Saved: faiss_index.bin rag_config.json rag_corpus.csv
Models offered: ['Llama 3.3 70B (fast)', 'GPT-OSS 120B (detailed, cited)']


In [13]:
# End-of-notebook checks: alignment, corpus filters, evaluation and exports are all consistent
assert len(metadata) == len(embeddings)
assert index.ntotal == len(corpus)
assert index.d == 384
assert corpus["word_count"].ge(MIN_WORDS).all()
assert len(retrieval_evaluation) >= 6
assert (RAG_DIR / "faiss_index.bin").exists()
assert (RAG_DIR / "rag_corpus.csv").exists()
assert (RAG_DIR / "rag_config.json").exists()
assert (TABLE_DIR / "rag_model_comparison.csv").exists()

print(f"RAG corpus: {len(corpus):,} evidence reviews from {corpus['game_name'].nunique()} games.")
print(f"Bi-encoder hit@5: {retrieval_evaluation['hit@5 (bi-encoder)'].mean():.0%}")
print(f"With reranking:   {retrieval_evaluation['hit@5 (+ reranker)'].mean():.0%}")
print("Eight questions is useful as a regression check but too small for a strong general claim.")
print("All RAG alignment, corpus and export checks passed.")

RAG corpus: 41,170 evidence reviews from 241 games.
Bi-encoder hit@5: 88%
With reranking:   88%
Eight questions is useful as a regression check but too small for a strong general claim.
All RAG alignment, corpus and export checks passed.
